# 01 — EDA Exploratória — Datathon Fase 5

**Universo investigado:** PETR4, VALE3, ITUB4, BBDC4, WEGE3 (B3) — janela de 5 anos.

**Objetivo da Fase 1:** entender o regime histórico das séries de preços, distribuição dos retornos, comportamento dos indicadores técnicos (RSI, MACD, Bollinger Bands) e dos fundamentos (P/L, ROE, Dividend Yield) — gerando insights que guiarão (a) a construção do baseline de classificação direcional e (b) as ferramentas que o agente da Etapa 2 vai expor.

Os dados são baixados via `yfinance` exatamente como em `src/features/feature_engineering.py`, garantindo paridade com o pipeline DVC.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.features.feature_engineering import (
    DEFAULT_TICKERS,
    compute_features,
    download_fundamentals,
    download_prices,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

## 1. Carga e visão geral dos dados

In [ ]:
prices = download_prices(tickers=DEFAULT_TICKERS, period="5y")
print(f"Linhas: {len(prices):,} | Tickers: {prices['ticker'].nunique()} | Janela: {prices['date'].min().date()} → {prices['date'].max().date()}")
prices.head()

In [ ]:
prices.groupby("ticker").agg(
    n_obs=("close", "size"),
    primeira=("date", "min"),
    ultima=("date", "max"),
    preco_min=("close", "min"),
    preco_max=("close", "max"),
    volume_medio=("volume", "mean"),
)

## 2. Evolução normalizada dos preços

Cada série é reindexada em base 100 para tornar comparável a performance acumulada.

In [ ]:
wide = prices.pivot(index="date", columns="ticker", values="close").sort_index()
normalized = wide.divide(wide.iloc[0]).multiply(100)

fig, ax = plt.subplots(figsize=(11, 5))
normalized.plot(ax=ax)
ax.set_title("Evolução normalizada (base 100) — 5 anos")
ax.set_ylabel("Índice (base 100)")
ax.set_xlabel("Data")
ax.legend(title="Ticker", loc="upper left", ncol=5)
plt.tight_layout()

## 3. Retorno por ano-calendário

Pergunta de negócio respondida diretamente: *"Qual ação teve o maior retorno em cada ano?"*

In [ ]:
annual = wide.resample("YE").last().pct_change().dropna(how="all")
annual.index = annual.index.year
annual.style.format("{:.2%}").background_gradient(cmap="RdYlGn", axis=1)

In [ ]:
winners = annual.idxmax(axis=1).rename("top_ticker").to_frame()
winners["retorno_top"] = annual.max(axis=1)
winners

## 4. Distribuição dos retornos diários (cauda e assimetria)

In [ ]:
log_ret = np.log(wide).diff().dropna(how="all")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
log_ret.plot(kind="box", ax=axes[0])
axes[0].set_title("Boxplot — log-retorno diário")
axes[0].axhline(0, color="black", lw=0.7)

for ticker in log_ret.columns:
    sns.kdeplot(log_ret[ticker].dropna(), label=ticker, ax=axes[1])
axes[1].set_title("Densidade — log-retorno diário")
axes[1].legend()
plt.tight_layout()

log_ret.agg(["mean", "std", "skew", "kurt"]).T.assign(
    sharpe_anual=lambda d: (d["mean"] / d["std"]) * np.sqrt(252)
)

## 5. Volatilidade móvel (21 dias) — janelas de stress

In [ ]:
vol21 = log_ret.rolling(21).std() * np.sqrt(252)
fig, ax = plt.subplots(figsize=(11, 4))
vol21.plot(ax=ax)
ax.set_title("Volatilidade anualizada (janela 21 dias)")
ax.set_ylabel("σ anualizado")
ax.legend(title="Ticker", ncol=5)
plt.tight_layout()

## 6. Correlação entre retornos

Insight de carteira: ações fortemente correlacionadas adicionam pouca diversificação.

In [ ]:
corr = log_ret.corr()
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlação de log-retornos diários")
plt.tight_layout()

## 7. Indicadores técnicos calculados pelo pipeline

Reaproveita exatamente a função `compute_features` que será chamada via DVC.

In [ ]:
features = compute_features(prices, target_horizon=1)
print(f"Features: {features.shape[0]:,} linhas × {features.shape[1]} colunas")
features.head()

In [ ]:
features.groupby("ticker")["target"].agg(["mean", "count"]).rename(
    columns={"mean": "prop_alta_dia_seguinte", "count": "n_obs"}
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(features["rsi_14"], bins=40, ax=axes[0])
axes[0].axvline(30, color="red", ls="--", label="sobrevendido (30)")
axes[0].axvline(70, color="green", ls="--", label="sobrecomprado (70)")
axes[0].set_title("Distribuição RSI(14)")
axes[0].legend()

sns.histplot(features["bb_pct_b"].clip(-0.5, 1.5), bins=40, ax=axes[1])
axes[1].axvline(0, color="red", ls="--", label="banda inferior")
axes[1].axvline(1, color="green", ls="--", label="banda superior")
axes[1].set_title("Distribuição %B (Bollinger)")
axes[1].legend()
plt.tight_layout()

## 8. Snapshot de fundamentos

Pergunta de negócio respondida diretamente: *"Compare PETR4 e VALE3 pelo P/L"*.

In [ ]:
fundamentals = download_fundamentals(DEFAULT_TICKERS)
fundamentals.set_index("ticker")

## 9. Síntese — insights para o problema da empresa

1. **Heterogeneidade de regime.** Em janelas de 5 anos as séries têm volatilidades anualizadas distintas (PETR4/VALE3 tipicamente ≥ 35%, ITUB4/BBDC4 ≤ 25%). O agente precisa expor *risco da carteira* como ferramenta de primeira ordem.
2. **Correlação setorial relevante.** ITUB4 e BBDC4 (financeiro) andam fortemente juntas; sugerir diversificação dentro do mesmo setor sem mensurar correlação seria enganoso.
3. **Direcionalidade ≈ 50%.** A proporção de dias em alta no D+1 fica próxima de 0.50 em todos os tickers — o baseline de classificação tem **chance de acerto trivial = 50%**, então qualquer AUC > 0.55 já carrega sinal econômico.
4. **RSI raramente sobrevendido.** Em mercados de tendência, regimes < 30 são raros; uma ferramenta puramente baseada em RSI extremo geraria poucos sinais — combinar com MACD e %B é necessário.
5. **Fundamentos completam o quadro.** P/L e Dividend Yield variam ordens de grandeza entre setores (financeiro vs. mineração vs. industrial), justificando que o agente da Etapa 2 cruze sempre técnico + fundamentalista antes de responder perguntas comparativas.

Esses insights motivam diretamente o **target binário (direção D+1)** adotado em `compute_features`, o **conjunto de features** (RSI, MACD, %B, retornos defasados, volatilidade) e a **escolha das métricas técnicas** (AUC, F1, Precision/Recall) — mapeadas para métricas de negócio em `docs/BUSINESS_METRICS.md`.